In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from environment import GridWorld
from value_function import DriveCORE
from agent import choose_action

grid = GridWorld(width=20, height=20, num_food=15, num_obstacles=30, num_players=1, seed=42)
drives = DriveCORE()
drives.record_visit(grid.players[0].row, grid.players[0].col)

In [ ]:
from gymnasium import gym


In [ ]:
N_STEPS = 300
energy_log = []
position_log = []

for _ in range(N_STEPS):
    p = grid.players[0]
    energy_log.append(p.energy)
    position_log.append((p.row, p.col))

    direction = choose_action(grid, 0, drives)
    grid.step(0, direction)
    drives.record_visit(grid.players[0].row, grid.players[0].col)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Gate 1 & 2: energy over time with critical threshold
CRITICAL = 0.2
axes[0].plot(energy_log, color='steelblue')
axes[0].axhline(CRITICAL, color='red', linestyle='--', label=f'critical ({CRITICAL})')
axes[0].set_title('Energy over time')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Energy')
axes[0].legend()

# Gate 2: urgency over time — shows nonlinear response to low energy
urgency_log = [drives.energy_drive.urgency(e) for e in energy_log]
axes[1].plot(urgency_log, color='orange')
axes[1].set_title('Energy urgency (nonlinear response)')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Urgency')

# Gate 3: visit heatmap — curiosity should produce broad coverage
visit_grid = np.zeros((grid.height, grid.width))
for (r, c), count in drives.curiosity_drive.visited.items():
    visit_grid[r, c] = count
im = axes[2].imshow(visit_grid, cmap='hot')
axes[2].set_title('Visit counts (curiosity coverage)')
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.show()